# 💊 Clinical-Grade Drug Interaction Knowledge Base — RAG Ingestion Pipeline  

<span style="color:red">by Ridwan Oladipo, MD | Medical AI Specialist</span>  

Production-grade ingestion pipeline unifying **191,541 DrugBank interactions** with **RxNorm mappings** into a clean, clinically reliable DDI knowledge base:  

- **Hierarchical normalization** (brand/generic/synonym → ingredient via RxCUI priority: IN > BN > PIN > SY)
- **Synonym unification** (1,369 acetaminophen variants → RxCUI 161)
- **Multi-tier resolution** (local cache → reverse index → RxNav API fallback)  
- **Crash-safe caching** (77,518 brand→ingredient links, 6,409 ingredient names)  
- **Atomic persistence** for multi-day API runs  
- **Bidirectional lookup architecture** (name↔RxCUI)  

🚀 Achieves **~90% pair-level mapping coverage**, yielding **170K+ verified interaction pairs** — powering RAG-based drug safety reasoning with clinical precision and production reliability.  

>⚕️ **Clinical safety meets engineering excellence** — resolving synonym chaos into a structured, high-trust pharmacologic graph.

## 📦 Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import pickle
import time
import re
from collections import defaultdict, Counter
from pathlib import Path

print("✅ Imports loaded")

✅ Imports loaded


## 📂 Load RxNorm Mappings

In [2]:
rxnorm_df = pd.read_csv('data/rxnorm_mappings.csv')

print(f"Shape: {rxnorm_df.shape}")
print(f"\nType distribution:\n{rxnorm_df['type'].value_counts()}")
print(f"\nSample:\n{rxnorm_df.head()}")

Shape: (178731, 4)

Type distribution:
type
DP            38607
SY            28324
PSN           21229
SCD           12033
SCDC          10146
SBD            8087
SU             7858
SBDC           7034
SBDG           6709
SCDG           6272
IN             5785
SCDF           5551
SBDF           5000
BN             4141
SCDGP          2480
SCDFP          1995
PIN            1895
SBDFP          1768
MTH_RXN_DP     1541
MIN             982
BPCK            696
GPCK            587
PT               11
Name: count, dtype: int64

Sample:
   rxcui type            name       name_norm
0     38   BN        Parlodel        parlodel
1     44   IN           mesna           mesna
2     44   SU           mesna           mesna
3     61   IN    beta-alanine    beta-alanine
4     61   SU  .BETA.-ALANINE  .beta.-alanine
